In [ ]:
# imports

import os
from contextlib import AsyncExitStack

import gradio as gr
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [ ]:
# constants

MODEL = "gpt-4o-mini"

WELCOME = (
    "Welcome to the Nigerian Commodity Market Assistant. "
    "Ask me to search for commodity news, manage your watchlist, or send you a price alert."
)

In [ ]:
# environment setup

load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY")
push_user  = os.getenv("PUSHOVER_USER")
push_token = os.getenv("PUSHOVER_TOKEN")

print("OPENAI_API_KEY :", "✅" if openai_key else "❌ MISSING")
print("SERPER_API_KEY :", "✅" if serper_key else "❌ MISSING")
print("PUSHOVER_USER  :", "✅" if push_user  else "❌ MISSING")
print("PUSHOVER_TOKEN :", "✅" if push_token else "❌ MISSING")

In [ ]:
# MCP server params
watchlist_params = {"command": "uv", "args": ["run", "watchlist_server.py"]}

serper_params = {"command": "uv", "args": ["run", "serper_server.py"]}

push_params = {"command": "uv", "args": ["run", "push_server.py"]}

ALL_PARAMS = [watchlist_params, serper_params, push_params]

In [ ]:
# Verify MCP tools

for params in ALL_PARAMS:
    async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        label = params["args"][-1] if params["command"] == "uv" else params["args"][1]
        print(f"{label}: {[t.name for t in tools]}")

In [ ]:
# Agent instructions

INSTRUCTIONS = """
You are a Nigerian Commodity Market Assistant. You help users stay informed about
commodity prices and news relevant to Nigerian and West African markets — including
grains (maize, sorghum, rice), produce (tomatoes, yam, cassava), and energy (crude oil).

You have access to three sets of tools:

WATCHLIST TOOLS (watchlist_server):
- add_to_watchlist: save a commodity the user wants to track
- get_watchlist: retrieve all tracked commodities
- remove_from_watchlist: remove a commodity from tracking
- get_current_date: get today's date for context

SEARCH TOOLS (serper_server):
- search_web: search the web for commodity news and prices

NOTIFICATION TOOLS (Pushover):
- push: send a push notification to the user's phone with a brief alert

GUIDELINES:
- Always check the current date before researching prices so your context is accurate.
- When the user asks about a commodity, search for the latest news and prices.
- If the user asks to track something, add it to the watchlist.
- If you find a significant price movement or market alert, offer to send a push notification.
- Keep responses concise and focused on the Nigerian/West African market context.
- Never invent prices — always search first.
"""


In [ ]:
# Helper functions

def add_user_message(message, history):
    history.append({"role": "user", "content": message})
    return "", history

def disable_btn():
    return gr.update(interactive=False)

def enable_btn():
    return gr.update(interactive=True)

In [ ]:
# Response function

async def get_response(history):
    message = history[-1]["content"]
    async with AsyncExitStack() as stack:
        servers = [
            await stack.enter_async_context(
                MCPServerStdio(params=p, client_session_timeout_seconds=60)
            )
            for p in ALL_PARAMS
        ]
        agent = Agent(
            name="CommodityAssistant",
            instructions=INSTRUCTIONS,
            model=MODEL,
            mcp_servers=servers,
        )
        with trace("CommodityAssistant"):
            result = await Runner.run(agent, message, max_turns=20)

    response = result.final_output
    history.append({"role": "assistant", "content": response})
    return "", history

In [ ]:
# Gradio UI

with gr.Blocks() as ui:
    gr.Markdown("## Nigerian Commodity Market Assistant")
    gr.Markdown(
        "Track commodities, search for market news, and receive price alerts — "
        "powered by MCP tools and grounded in West African market context."
    )

    chatbot = gr.Chatbot(
        value=[{"role": "assistant", "content": WELCOME}],
        type="messages",
        label="Commodity Assistant",
        height=500,
    )
    question_box = gr.Textbox(
        placeholder="e.g. What is the latest price of maize in Nigeria?",
        label="Your question",
    )
    submit_btn = gr.Button("Send", variant="primary")

    submit_btn.click(
        fn=disable_btn, outputs=submit_btn
    ).then(
        fn=add_user_message, inputs=[question_box, chatbot], outputs=[question_box, chatbot]
    ).then(
        fn=get_response, inputs=[chatbot], outputs=[question_box, chatbot]
    ).then(
        fn=enable_btn, outputs=submit_btn
    )

ui.launch(inbrowser=True)